# BLAST Tool Examples

This notebook demonstrates two BLAST tools:
- **`run_blast_search`** -- Unified BLAST search supporting both online (NCBI) and local modes
- **`run_create_blast_db`** -- Create a local BLAST database from a FASTA file

## Imports

In [1]:
from bio_programming_tools import (
    BlastSearchConfig,
    BlastSearchInput,
    CreateBlastDbConfig,
    CreateBlastDbInput,
    run_blast_search,
    run_create_blast_db,
)
from pathlib import Path
import tempfile

## 1. Create a Local BLAST Database

First, let's create a small FASTA file with some protein sequences and build a local BLAST database from it. This triggers `ToolInstance("blast")` which auto-downloads BLAST+ binaries on first run.

### API Reference

**`CreateBlastDbInput`**

| Field | Type | Description |
|-------|------|-------------|
| `fasta` | `str` | Path to the input FASTA file containing sequences to create a BLAST database from |

**`CreateBlastDbConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `dbtype` | `Literal["nucl", "prot"]` | `"nucl"` | Specifies the type of database to create: Nucleotide or Protein |
| `out_prefix` | `Optional[str]` | `None` | File-path prefix for database files (default: FASTA stem) |
| `title` | `Optional[str]` | `None` | Optional name for the database |

**`CreateBlastDbOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `db_path` | `str` | Path to the generated BLAST database |

In [2]:
# Create a temp directory for our test database
tmp_dir = tempfile.mkdtemp(prefix="blast_example_")

# Write a small FASTA with well-known protein sequences
fasta_path = Path(tmp_dir) / "test_sequences.fasta"
fasta_path.write_text("""\
>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH
GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL
LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR
>sp|P68871|HBB_HUMAN Hemoglobin subunit beta
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST
PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP
ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH
>sp|P02144|MYG_HUMAN Myoglobin
MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL
KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP
VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG
""")

print(f"Wrote test FASTA to: {fasta_path}")

# Create the BLAST database
db_input = CreateBlastDbInput(fasta=str(fasta_path))
db_config = CreateBlastDbConfig(dbtype="prot", title="Test Protein DB")

db_result = run_create_blast_db(db_input, db_config)
print(f"Database created at: {db_result.db_path}")

Wrote test FASTA to: /var/folders/rs/6dqw0_k1125fl7f7_9h85hgh0000gn/T/blast_example__5l1q3lc/test_sequences.fasta
Database created at: /var/folders/rs/6dqw0_k1125fl7f7_9h85hgh0000gn/T/blast_example__5l1q3lc/test_sequences


## 2. Local BLAST Search

Search a query protein against our local database using `search_mode="local"`. The query is a fragment of hemoglobin beta -- it should match HBB_HUMAN with high identity.

### API Reference

**`BlastSearchInput`**

| Field | Type | Description |
|-------|------|-------------|
| `query` | `str` | Query sequence string or path to a FASTA file (auto-detected) |
| `query_type` | `str` | Automatically inferred: `"sequence"` or `"fasta_path"` |

**`BlastSearchConfig`** (key parameters shown)

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `search_mode` | `Literal["online", "local"]` | `"online"` | Search mode |
| `program` | `Literal["blastn", "blastp", "blastx", "tblastn", "tblastx"]` | `"blastn"` | BLAST program |
| `local_db` | `Optional[str]` | `None` | Path to local BLAST database (required for local mode) |
| `database` | `str` | `"nt"` | NCBI database (online mode) |
| `evalue` | `Optional[float]` | `None` | E-value threshold |
| `num_threads` | `int` | `4` | CPU threads (local mode) |

**`BlastSearchOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `results_df` | `Optional[pd.DataFrame]` | DataFrame with BLAST results (columns: `qseqid`, `sseqid`, `pident`, `length`, `mismatch`, `gapopen`, `qstart`, `qend`, `sstart`, `send`, `evalue`, `bitscore`) |
| `num_hits` | `int` | Number of BLAST hits found |

In [3]:
# Hemoglobin beta sequence — used for both local and online searches
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

# Write as FASTA for local BLAST
query_path = Path(tmp_dir) / "query.fasta"
query_path.write_text(f">query_hbb\n{query_seq}\n")

# Run local BLAST search
search_input = BlastSearchInput(query=str(query_path))
search_config = BlastSearchConfig(
    search_mode="local",
    program="blastp",
    local_db=db_result.db_path,
    num_threads=2,
)

result = run_blast_search(search_input, search_config)
print(f"Found {result.num_hits} hits")
result.results_df

Found 2 hits


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,query_hbb,sp|P68871|HBB_HUMAN,100.000,147,0,0,1,147,1,147,1.390000e-111,301
1,query_hbb,sp|P69905|HBA_HUMAN,43.448,145,74,3,4,146,3,141,5.420000e-38,114


## 3. Online BLAST Search

Search NCBI's online databases directly with a protein sequence using `search_mode="online"` (the default). Requires an internet connection and may take a few minutes.

The query can be a raw sequence string — no FASTA file needed for online mode.

In [4]:
# Use the same hemoglobin beta sequence as the local search
inputs = BlastSearchInput(query=query_seq)
config = BlastSearchConfig(
    search_mode="online",
    program="blastp",
    database="swissprot",
    hitlist_size=5,
)

online_result = run_blast_search(inputs, config)
print(f"Found {online_result.num_hits} hits")
online_result.results_df

Found 5 hits


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,unnamed,P68871,100.000000,147,0,0,1,147,1,147,5.822150e-106,301.212
1,unnamed,P02024,99.319728,147,1,0,1,147,1,147,2.034720e-105,300.056
2,unnamed,P02025,98.630137,146,2,0,2,147,1,146,2.964790e-103,294.278
3,unnamed,P02032,97.260274,146,4,0,2,147,1,146,2.438800e-102,291.967
4,unnamed,P19885,95.918367,147,6,0,1,147,1,147,4.274080e-102,291.582


## 4. Export Results

Export results to CSV files for downstream analysis.

- **CSV** -- Tabular format with all BLAST hit fields
- **JSON** -- JSON format with records orientation

In [5]:
output_dir = Path("./example_output")

result.export(name="blast_search_local_results", export_path=output_dir, file_format="csv")
online_result.export(name="blast_search_online_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
